In [ ]:
import numpy as np
import matplotlib.pyplot as plt


NUM_ELEMENTS = 32
NUM_LOCAL_FREQ = 672
SAMPLES_PER_DATA_SET = 14976

NR_POLARIZATIONS = 2
NR_RECEIVERS = NUM_ELEMENTS // NR_POLARIZATIONS   # 16
NUM_BASELINES = NR_RECEIVERS * (NR_RECEIVERS + 1) // 2   # 136

FRAME_IDX = 0  # Usa el mismo índice para voltage y correlaciones
VOLTAGE_PATH = f"/home/juan_pablo/kotekan/test_data/frame_{FRAME_IDX}.bin"
GPU_PATH = f"/home/juan_pablo/kotekan/test_corr_gpu/frame_{FRAME_IDX}.bin"

In [ ]:
def load_voltage_frame(path):
    raw = np.fromfile(path, dtype=np.uint8)
    return raw.reshape(SAMPLES_PER_DATA_SET, NUM_LOCAL_FREQ, NUM_ELEMENTS)  # [t,f,e]


def load_gpu_corr_frame(path):
    raw = np.fromfile(path, dtype=np.int32)
    corr_i32 = raw.reshape(
        NUM_LOCAL_FREQ,
        NUM_BASELINES,
        NR_POLARIZATIONS,
        NR_POLARIZATIONS,
        2
    )

    corr = corr_i32[..., 0].astype(np.int64) + 1j * corr_i32[..., 1].astype(np.int64)
    return corr_i32, corr  # [f, baseline, polY, polX]


def unpack(u8):
    real = (u8 >> 4).astype(np.int8)
    imag = (u8 & 0x0F).astype(np.int8)
    real[real >= 8] -= 16
    imag[imag >= 8] -= 16
    return real.astype(np.int16) + 1j * imag.astype(np.int16)


def baseline_index(recv_y, recv_x):
    return recv_y * (recv_y + 1) // 2 + recv_x


def elem_to_recv_pol(a):
    recv = a // 2
    pol = a % 2
    return recv, pol

# Same Layout as GPU: [f, baseline, polY, polX]
def build_cpu_corr_in_gpu_layout(voltage_u8):
    x = unpack(voltage_u8)                     # [t,f,e]
    x = np.transpose(x, (1, 2, 0))            # [f,e,t]
    x = np.ascontiguousarray(x)
    x = x.reshape(NUM_LOCAL_FREQ, NR_RECEIVERS, NR_POLARIZATIONS, SAMPLES_PER_DATA_SET) #x[f, recv, pol, t]

    cpu_corr = np.zeros(
        (NUM_LOCAL_FREQ, NUM_BASELINES, NR_POLARIZATIONS, NR_POLARIZATIONS),
        dtype=np.complex128
    )

    for recv_y in range(NR_RECEIVERS):
        Y = x[:, recv_y, :, :]

        for recv_x in range(recv_y + 1):
            X = x[:, recv_x, :, :]

            vis = np.einsum("fpt,fqt->fpq", Y, np.conj(X), optimize=True) # to not loss precision

            b = baseline_index(recv_y, recv_x)
            cpu_corr[:, b, :, :] = vis

    return cpu_corr



def extract_pair_from_gpu_layout(corr_gpu_layout, a, b):
    recv_a, pol_a = elem_to_recv_pol(a)
    recv_b, pol_b = elem_to_recv_pol(b)

    if recv_b <= recv_a:
        bidx = baseline_index(recv_a, recv_b)
        return corr_gpu_layout[:, bidx, pol_a, pol_b]
    else:
        bidx = baseline_index(recv_b, recv_a)
        return np.conj(corr_gpu_layout[:, bidx, pol_b, pol_a])



In [ ]:

voltage = load_voltage_frame(VOLTAGE_PATH)         # [t,f,e]
gpu_corr_i, gpu_corr = load_gpu_corr_frame(GPU_PATH)
cpu_corr = build_cpu_corr_in_gpu_layout(voltage)





In [ ]:
# Mantener consistente con FRAME_IDX definido en la celda inicial
gpu_corr_i, gpu_corr = load_gpu_corr_frame(GPU_PATH)
print("GPU_PATH usado:", GPU_PATH)

In [ ]:
print("voltage shape:", voltage.shape)             # (15360, 672, 32)
print("gpu_corr shape:", gpu_corr.shape)          # (672, 136, 2, 2)
print("cpu_corr shape:", cpu_corr.shape)          # (672, 136, 2, 2)


abs_diff = np.abs(cpu_corr - gpu_corr)
print("max |CPU-GPU|:", abs_diff.max())
print("mean |CPU-GPU|:", abs_diff.mean())

In [ ]:
ANT_REF = 16
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(8, 32)]   # 8..31 inclusive

frequencies = np.linspace(300, 501.6, 672, endpoint=False)


n_cols = 4
n_plots = len(PAIRS_TO_PLOT)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows), sharex=False, sharey=False)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr, a, b)

    amp_diff = np.abs(cpu_pair) - np.abs(gpu_pair)

    ax = axes_flat[idx]
    ax.scatter(frequencies, amp_diff, s=2)
    ax.set_title(f"Amplitude diff ({a},{b})", fontsize=10)
    ax.set_xlabel("Frequency [MHz]", fontsize=8)
    ax.set_ylabel("Amplitude diff", fontsize=8)
    ax.grid(True, alpha=0.5)

# apagar ejes sobrantes
for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Amplitude difference CPU-GPU for antenna {ANT_REF}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
ANT_REF = 16
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(8, 32)]   # 8..31 inclusive

frequencies = np.linspace(300, 501.6, 672, endpoint=False)


n_cols = 4
n_plots = len(PAIRS_TO_PLOT)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows), sharex=False, sharey=False)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr, a, b)

    phase_diff = np.angle(cpu_pair * np.conj(gpu_pair))

    ax = axes_flat[idx]
    ax.scatter(frequencies, phase_diff, s=5)
    ax.set_title(f"Phase diff ({a},{b})", fontsize=10)
    ax.set_xlabel("Frequency [MHz]", fontsize=8)
    ax.set_ylabel("Phase diff [rad]", fontsize=8)
    ax.grid(True, alpha=0.5)

# apagar ejes sobrantes
for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Phase difference CPU-GPU for antenna {ANT_REF}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
ANT_REF = 14
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(8, 32)]   # 8..31 inclusive

frequencies = np.linspace(300, 501.6, 672, endpoint=False)


n_cols = 4
n_plots = len(PAIRS_TO_PLOT)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows), sharex=False, sharey=False)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr, a, b)


    ax = axes_flat[idx]
    ax.scatter(frequencies, np.abs(cpu_pair), label="CPU", s=5)
    ax.scatter(frequencies, np.abs(gpu_pair), label="GPU", s=3)
    ax.set_title(f"Amplitude ({a},{b})", fontsize=10)
    ax.set_xlabel("Frequency [MHz]", fontsize=8)
   # ax.set_ylim(-10, 500)
    
    ax.grid(True, alpha=0.5)
    ax.legend()

# apagar ejes sobrantes
for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Amp CPU-GPU for fixed antenna {ANT_REF}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
def ant_to_adc_label(ant):
    if 24 <= ant <= 31:
        adc = "A"
        col = ant - 24
    elif 16 <= ant <= 23:
        adc = "B"
        col = ant - 16
    elif 8 <= ant <= 15:
        adc = "C"
        col = ant - 8
    elif 0 <= ant <= 7:
        adc = "D"
        col = ant
    else:
        raise ValueError(f"Antena fuera de rango: {ant}")

    return f"{adc}{col}"

i = 16
n_ant = 32

frequencies = np.linspace(300, 501.6, 672, endpoint=False)

n_cols = 4
n_pairs = len(range(8, n_ant))
n_rows = int(np.ceil(n_pairs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()

print(f"Calculating GPU correlations and plotting for antenna i={i}...")

plot_idx = 0
for j in range(8, n_ant):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, i, j)
    phase = np.angle(cpu_pair)

    label_i = ant_to_adc_label(i)
    label_j = ant_to_adc_label(j)

    ax = axes_flat[plot_idx]
    ax.scatter(frequencies, phase, s=3)
    ax.set_title(f"{label_i} vs {label_j}", fontsize=10)

    if plot_idx >= n_pairs - n_cols:
        ax.set_xlabel("Frequency channel", fontsize=8)

    if plot_idx % n_cols == 0:
        ax.set_ylabel("Phase [rad]", fontsize=8)

    ax.grid(True, alpha=0.3)

    plot_idx += 1

# ocultar ejes vacíos
for k in range(plot_idx, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Phase spectra for fixed antenna {i} ({ant_to_adc_label(i)}) vs antennas 8..31 on CPU",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
import re
from pathlib import Path


RAW_NET_DIR = Path('/data/corr/network')
RAW_CORR_DIR = Path('/data/corr/host_corr')
CPU_NET_FRAME_IDX = 8 

def frame_idx_from_name(p: Path):
    m = re.search(r'_(\d+)\.bin$', p.name)
    return int(m.group(1)) if m else None

def load_rawfile_payload_bytes(path):
    raw = np.fromfile(path, dtype=np.uint8)
    meta_size = int(np.frombuffer(raw[:4].tobytes(), dtype=np.uint32)[0]) # 4 bytes of metadata size
    off = 4 + meta_size
    return raw[off:]

def load_network_payload_from_raw(path):
    payload = load_rawfile_payload_bytes(path)
    return payload.reshape(SAMPLES_PER_DATA_SET, NUM_LOCAL_FREQ, NUM_ELEMENTS)

def load_corr_payload_from_raw(path):
    payload = load_rawfile_payload_bytes(path)
    raw_i32 = np.frombuffer(payload.tobytes(), dtype=np.int32)
    corr_i32 = raw_i32.reshape(NUM_LOCAL_FREQ, NUM_BASELINES, NR_POLARIZATIONS, NR_POLARIZATIONS, 2)
    corr = corr_i32[..., 0].astype(np.int64) + 1j * corr_i32[..., 1].astype(np.int64)
    return corr_i32, corr

net_files = sorted(RAW_NET_DIR.glob('*.bin'), key=frame_idx_from_name)
corr_files = sorted(RAW_CORR_DIR.glob('*.bin'), key=frame_idx_from_name)
net_map = {frame_idx_from_name(p): p for p in net_files}
corr_map = {frame_idx_from_name(p): p for p in corr_files}

print('Network frames:', sorted(net_map.keys()))
print('Corr frames   :', sorted(corr_map.keys()))


print(f'\nCalculando correlación CPU desde network frame {CPU_NET_FRAME_IDX}...')
voltage_raw = load_network_payload_from_raw(net_map[CPU_NET_FRAME_IDX])
cpu_corr_raw = build_cpu_corr_in_gpu_layout(voltage_raw)

rows = []
for gidx in sorted(corr_map.keys()):
    _, gpu_corr_raw = load_corr_payload_from_raw(corr_map[gidx])
    abs_diff = np.abs(cpu_corr_raw - gpu_corr_raw)
    mean_abs = float(abs_diff.mean())
    max_abs = float(abs_diff.max())

    mask = (np.abs(cpu_corr_raw) > 1e-9) & (np.abs(gpu_corr_raw) > 1e-9)
    if np.any(mask):
        dphi = np.angle(cpu_corr_raw[mask] * np.conj(gpu_corr_raw[mask]))
        mean_abs_dphi = float(np.mean(np.abs(dphi)))
    else:
        mean_abs_dphi = np.nan

    rows.append((gidx, mean_abs, max_abs, mean_abs_dphi))

rows = sorted(rows, key=lambda r: r[1])
print('\nRanking (CPU network frame fijo vs corr frames):')
for r in rows:
    print(f'corr_frame={r[0]:2d}  mean|diff|={r[1]:.6f}  max|diff|={r[2]:.1f}  mean|dphi|={r[3]:.6e} rad')

best = rows[0]
print(f'\nMEJOR MATCH: network frame {CPU_NET_FRAME_IDX} -> corr frame {best[0]}')

plt.figure(figsize=(10,4))
plt.plot([r[0] for r in rows], [r[1] for r in rows], marker='o')
plt.xlabel('corr frame index')
plt.ylabel('mean |CPU-GPU|')
plt.title(f'Network frame {CPU_NET_FRAME_IDX} vs corr frames (rawFileWrite payload)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Usar explícitamente el mejor match encontrado en la celda anterior
best_corr_idx = int(best[0])
_, gpu_corr_raw = load_corr_payload_from_raw(corr_map[best_corr_idx])
print(f"Plot de amplitud para best match: network frame {CPU_NET_FRAME_IDX} vs corr frame {best_corr_idx}")

ANT_REF = 13
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(8, 32)]   # 8..31 inclusive

frequencies = np.linspace(300, 501.6, 672, endpoint=False)


n_cols = 4
n_plots = len(PAIRS_TO_PLOT)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows), sharex=False, sharey=False)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr_raw, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr_raw, a, b)


    ax = axes_flat[idx]
    ax.scatter(frequencies, np.abs(cpu_pair), label="CPU", s=15)
    ax.scatter(frequencies, np.abs(gpu_pair), label="GPU", s=3)
    ax.set_title(f"Amplitude ({a},{b})", fontsize=10)
    ax.set_xlabel("Frequency [MHz]", fontsize=8)
    ax.grid(True, alpha=0.5)
    ax.legend()

# apagar ejes sobrantes
for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Amp CPU-GPU (best match) fixed antenna {ANT_REF} | net={CPU_NET_FRAME_IDX}, corr={best_corr_idx}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# Usar explícitamente el mejor match encontrado en la celda anterior
best_corr_idx = int(best[0])
_, gpu_corr_raw = load_corr_payload_from_raw(corr_map[best_corr_idx])
print(f"Plot de amplitud para best match: network frame {CPU_NET_FRAME_IDX} vs corr frame {best_corr_idx}")

ANT_REF = 16
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(8, 32)]   # 8..31 inclusive

frequencies = np.linspace(300, 501.6, 672, endpoint=False)


n_cols = 4
n_plots = len(PAIRS_TO_PLOT)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows), sharex=False, sharey=False)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr_raw, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr_raw, a, b)

    phase_diff = np.angle(cpu_pair * np.conj(gpu_pair))

    ax = axes_flat[idx]
    ax.scatter(frequencies, phase_diff, label="Phase Diff", s=15)
    ax.set_title(f"Phase Diff ({a},{b})", fontsize=10)
    ax.set_xlabel("Frequency [MHz]", fontsize=8)
    ax.grid(True, alpha=0.5)
    ax.legend()

# apagar ejes sobrantes
for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Phase Diff CPU-GPU (best match) fixed antenna {ANT_REF} | net={CPU_NET_FRAME_IDX}, corr={best_corr_idx}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# Fase sin mascara vs fase con mascara por amplitud (CPU vs GPU)
ANT_REF = 16
PAIR_START = 8
PAIR_END = 32   # exclusivo
AMP_PERCENTILE = 70  # mascara: conservar el 30% de mayor amplitud
EPS = 1e2

best_corr_idx = int(best[0])
_, corr_gpu_plot = load_corr_payload_from_raw(corr_map[best_corr_idx])
corr_cpu_plot = cpu_corr_raw  # ya está en formato [f, baseline, polY, polX]

pairs = [(ANT_REF, b) for b in range(PAIR_START, PAIR_END)]
frequencies = np.linspace(300, 501.6, NUM_LOCAL_FREQ, endpoint=False)

n_cols = 4
n_plots = len(pairs)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3.8 * n_rows), sharex=False, sharey=False)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(pairs):
    cpu_pair = extract_pair_from_gpu_layout(corr_cpu_plot, a, b)
    gpu_pair = extract_pair_from_gpu_layout(corr_gpu_plot, a, b)

    # Fase diferencial sin mascara
    phase_raw = np.angle(cpu_pair * np.conj(gpu_pair))

    # Mascara por amplitud: elimina bins con amplitud baja (fase inestable)
    amp_ref = np.minimum(np.abs(cpu_pair), np.abs(gpu_pair))
    thr = np.percentile(amp_ref, AMP_PERCENTILE)
    mask = amp_ref > max(thr, EPS)

    phase_masked = np.full_like(phase_raw, np.nan, dtype=np.float64)
    phase_masked[mask] = phase_raw[mask]

    ax = axes_flat[idx]
   # ax.scatter(frequencies, phase_raw, s=12, alpha=0.35, label='Sin mascara')
    ax.scatter(frequencies, phase_masked, s=5, alpha=0.85, label='Con mascara')
    ax.set_title(f'Phase diff ({a},{b})', fontsize=10)
    ax.set_xlabel('Frequency [MHz]', fontsize=8)
    ax.set_ylabel('Phase diff [rad]', fontsize=8)
    ax.grid(True, alpha=0.35)
    ax.legend(fontsize=7)

for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis('off')

fig.suptitle(
    f'CPU-GPU phase diff: sin mascara vs con mascara | ANT_REF={ANT_REF} | {source_tag}',
    fontsize=15
)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()